In [4]:
import matplotlib.pyplot as plt
import anndata
import scanpy as sc
import snapatac2 as snap
import numpy as np
import pandas as pd
import os
import scanpy.external as sce
import seaborn as sns
from sklearn.metrics import silhouette_score
import numpy as np
from scipy.stats import chi2


In [2]:
import warnings
warnings.filterwarnings("ignore")


In [ ]:
df_dmr= pd.read_csv('/data1st2/hannan_25/data/Nanopore_processV1/nanopore_08_differential/summary/dmrmerged_seg_anno_2tools_nofilter_0901.csv',index_col=0)
df_dmr


In [9]:
for mod in ['5mC', '5hmC']:
    df_dmr_5mc = df_dmr[df_dmr['mod']==mod]
    df_dmr_5mc['Region'] = df_dmr_5mc['comparision'].str[3:6]
    df_dmr_5mc = df_dmr_5mc#[df_dmr_5mc['Region']=='AMY']
    df_dmr_5mc_expanded = df_dmr_5mc.dmr.str.split('[|_:-]', expand=True)
    df_dmr_5mc_expanded.columns = ['methtype', 'motif2', 'chr','start', 'end']
    # Expand DMRs with length <100 to 100bp centered at original center
    df_dmr_5mc_expanded['length'] = df_dmr_5mc_expanded['end'].astype(int) - df_dmr_5mc_expanded['start'].astype(int)
    df_dmr_5mc_expanded['center'] = (df_dmr_5mc_expanded['end'].astype(int) + df_dmr_5mc_expanded['start'].astype(int)) //2
    df_dmr_5mc_expanded['start_expanded'] = df_dmr_5mc_expanded['center'] - 50
    df_dmr_5mc_expanded['end_expanded'] = df_dmr_5mc_expanded['center'] + 50
    # if legthn <100, expand start = start_expanded, end = end_expanded
    df_dmr_5mc_expanded.loc[df_dmr_5mc_expanded['length']<100, 'start'] = df_dmr_5mc_expanded.loc[df_dmr_5mc_expanded['length']<100, 'start_expanded']
    df_dmr_5mc_expanded.loc[df_dmr_5mc_expanded['length']<100, 'end'] = df_dmr_5mc_expanded.loc[df_dmr_5mc_expanded['length']<100, 'end_expanded']

    df_dmr_5mc =  pd.concat([df_dmr_5mc, df_dmr_5mc_expanded], axis=1)
    df_dmr_5mc.to_csv(f'/data2st1/junyi/output/atac1112/cCRE/{mod}_annotation.csv')
    df_bed= df_dmr_5mc.loc[:,['chr', 'start', 'end']].drop_duplicates()
    df_bed['chr'] = 'chr' + df_bed['chr'].astype(str)
    df_bed.to_csv(f'/data2st1/junyi/output/atac1112/cCRE/dmr_{mod}.bed', sep='\t', header=False, index=False)


In [47]:
df_bed.drop_duplicates(['chr', 'start', 'end'])

,chr,start,end
FC-AMY_vs_FW-AMY:methylDMR.1,chr1,100000185,100000671
FC-AMY_vs_FW-AMY:methylDMR.2,chr1,100213290,100213390
FC-AMY_vs_FW-AMY:methylDMR.3,chr1,100224782,100224882
FC-AMY_vs_FW-AMY:methylDMR.4,chr1,100226519,100226652
FC-AMY_vs_FW-AMY:methylDMR.5,chr1,100269082,100269242
...,...,...,...
297040,chrX,170011131,170011231
297052,chrX,170018649,170018749
297063,chrX,170019104,170019204
297083,chrY,90785820,90785920


In [10]:
adata_concat = snap.read_dataset('/data2st1/junyi/output/atac0627/doublet_filtered.h5ads/_dataset.h5ads')

In [11]:
%time hm5c_mat = snap.pp.make_peak_matrix(adata_concat,peak_file='/data2st1/junyi/output/atac1112/cCRE/dmr_5hmC.bed')
hm5c_mat.write(f"output/atac1112/3REGIONS_5hmc_new.h5ads")

... storing 'sample' as categorical
... storing 'celltype.L1' as categorical
... storing 'celltype.L2' as categorical
... storing 'Neurotransmitter_celltype' as categorical
... storing 'celltype.L1_ct' as categorical
... storing 'Sample_name' as categorical
... storing 'Condition' as categorical
... storing 'Region' as categorical
... storing 'celltype.L2.raw' as categorical
... storing 'region_nt' as categorical
... storing 'celltype.L3' as categorical
... storing 'celltype.L4' as categorical
... storing 'celltype.L2.refined' as categorical


CPU times: user 1h 57min, sys: 24min 12s, total: 2h 21min 13s
Wall time: 6min 14s


In [12]:
%time m5c_mat = snap.pp.make_peak_matrix(adata_concat,peak_file='/data2st1/junyi/output/atac1112/cCRE/dmr_5mC.bed')
m5c_mat.write(f"output/atac1112/3REGIONS_5mc_new.h5ads")
adata_concat.close()

... storing 'sample' as categorical
... storing 'celltype.L1' as categorical
... storing 'celltype.L2' as categorical
... storing 'Neurotransmitter_celltype' as categorical
... storing 'celltype.L1_ct' as categorical
... storing 'Sample_name' as categorical
... storing 'Condition' as categorical
... storing 'Region' as categorical
... storing 'celltype.L2.raw' as categorical
... storing 'region_nt' as categorical
... storing 'celltype.L3' as categorical
... storing 'celltype.L4' as categorical
... storing 'celltype.L2.refined' as categorical


CPU times: user 1h 57min 15s, sys: 23min 41s, total: 2h 20min 56s
Wall time: 6min 18s


In [64]:
annodict= {}
for methtype in ['5mC', '5hmC']:
    if methtype == '5mC':
        annodict[methtype] = pd.read_csv('/data2st1/junyi/output/atac1112/cCRE/5mC_annotation.csv',index_col=0)
    else:
        annodict[methtype] = pd.read_csv('/data2st1/junyi/output/atac1112/cCRE/5hmC_annotation.csv',index_col=0)
           


In [66]:
adata_region.var

,mod,motif,dmr,ifdifferent,score,num_sites,effect_size,case1_sig,case2_sig,diff.Methy,...,Region,methtype,motif2,chr,start,end,length,center,start_expanded,end_expanded
chr1:10004351-10004481,5mC,CG,5mC|CG_1:10004351-10004481,different,14.464736,3,0.164570,0.449213,0.284643,0.164570,...,AMY,5mC,CG,1,10004351,10004481,130,10004416,10004366,10004466
chr1:100076128-100076228,5mC,CG,5mC|CG_1:100076174-100076182,different,9.867896,3,0.139322,0.543241,0.403919,0.139322,...,AMY,5mC,CG,1,100076128,100076228,8,100076178,100076128,100076228
chr1:101169507-101169607,5mC,CG,5mC|CG_1:101169521-101169593,different,9.601184,3,0.182441,0.626671,0.444229,0.182441,...,AMY,5mC,CG,1,101169507,101169607,72,101169557,101169507,101169607
chr1:10156887-10156987,5mC,CG,5mC|CG_1:10156904-10156971,different,10.158190,3,0.112174,0.453135,0.340961,0.112174,...,AMY,5mC,CG,1,10156887,10156987,67,10156937,10156887,10156987
chr1:10170564-10170763,5mC,CG,5mC|CG_1:10170564-10170763,different,45.752996,11,0.160247,0.446176,0.285929,0.160247,...,AMY,5mC,CG,1,10170564,10170763,199,10170663,10170613,10170713
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
chrX:170009810-170009910,5mC,CN,5mC|CN_X:170009858-170009863,different,28.063795,2,0.121442,0.008890,0.007676,0.001214,...,AMY,5mC,CN,X,170009810,170009910,5,170009860,170009810,170009910
chrX:170010644-170010744,5mC,CN,5mC|CN_X:170010672-170010717,different,116.699443,18,0.108205,0.007810,0.006727,0.001082,...,AMY,5mC,CN,X,170010644,170010744,45,170010694,170010644,170010744
chrX:170010999-170011099,5mC,CN,5mC|CN_X:170011039-170011059,different,119.050779,12,0.090263,0.008190,0.007287,0.000903,...,AMY,5mC,CN,X,170010999,170011099,20,170011049,170010999,170011099
chrX:170011131-170011231,5mC,CN,5mC|CN_X:170011176-170011186,different,82.541595,6,0.156818,0.007107,0.005539,0.001568,...,AMY,5mC,CN,X,170011131,170011231,10,170011181,170011131,170011231


In [67]:
for methtype in ['5mC', '5hmC']:
    methtypeL = methtype.lower()
    adata = sc.read_h5ad(f"output/atac1112/3REGIONS_{methtypeL}_new.h5ads")
    dups = adata.var_names.duplicated()
    # Drop duplicated genes
    dmr_mat = adata[:, ~dups].copy()
    #dmr_mat = dmr_mat.obs[~((dmr_mat.obs['celltype.L1_ct']=='OPC') & (dmr_mat.obs['Neurotransmitter_celltype']!='NN'))]
    dmr_mat = dmr_mat[~((dmr_mat.obs['celltype.L1_ct']=='OPC') & (dmr_mat.obs['Neurotransmitter_celltype']!='NN'))]
    adata_dict={}
    for region in ['AMY', 'HIP', 'PFC']:
        df_anno = annodict[methtype]
        df_region = df_anno[df_anno['Region']==region]
        df_region['dmr_name'] = 'chr'+df_region['chr'].astype(str)+':'+df_region['start'].astype(str)+'-'+df_region['end'].astype(str)
        adata_region = dmr_mat[dmr_mat.obs['Region']==region,]
        df_dmr_ano = df_region.drop_duplicates(subset=['dmr_name'])
        df_dmr_ano.drop('gene',axis=1,inplace=True)
        df_dmr_ano.set_index('dmr_name', inplace=True)
        adata_region = adata_region[:, df_dmr_ano.index]
        adata_region.var= df_dmr_ano.loc[adata_region.var_names, :]
        adata_region.var['chr'] = "chr"+adata_region.var['chr'].astype(str)
        #print(region, df_region.shape[0])
        adata_region.write_h5ad(f"/data2st1/junyi/output/atac1112/subset/dmr_region_nt/{region}_{methtype}.h5ad")


... storing 'mod' as categorical
... storing 'motif' as categorical
... storing 'ifdifferent' as categorical
... storing 'comparision' as categorical
... storing 'tools' as categorical
... storing 'anno' as categorical
... storing 'Region' as categorical
... storing 'methtype' as categorical
... storing 'motif2' as categorical
... storing 'chr' as categorical
... storing 'mod' as categorical
... storing 'motif' as categorical
... storing 'ifdifferent' as categorical
... storing 'comparision' as categorical
... storing 'tools' as categorical
... storing 'anno' as categorical
... storing 'Region' as categorical
... storing 'methtype' as categorical
... storing 'motif2' as categorical
... storing 'chr' as categorical
... storing 'mod' as categorical
... storing 'motif' as categorical
... storing 'ifdifferent' as categorical
... storing 'comparision' as categorical
... storing 'tools' as categorical
... storing 'anno' as categorical
... storing 'Region' as categorical
... storing 'methtype

In [14]:
def g_test_row(row):
    row = np.array(row, dtype=float)
    total = row.sum()
    expected = np.repeat(total/len(row), len(row))

    # Avoid log(0)
    row_safe = np.where(row > 0, row, 1e-12)

    G = 2 * np.sum(row_safe * np.log(row_safe / expected))
    pval = 1 - chi2.cdf(G, df=len(row)-1)
    return pval

for methtype in ['5mC', '5hmC']:
    for region in ['AMY', 'HIP', 'PFC']:
        adata_region = sc.read_h5ad(f"/data2st1/junyi/output/atac1112/subset/dmr_region_nt/{region}_{methtype}.h5ad")
        adata_region.layers['counts'] = adata_region.X
        adata_region.X = adata_region.layers['counts']
        sc.pp.normalize_total(adata_region, target_sum=1e6)
        #sc.pp.log1p(adata_region)
        group_key = "celltype.L1_ct"  # 替换为 obs 的列名，例如 "cell_type"
        agg_adata = sc.get.aggregate(adata_region,by=group_key,func='mean')
        agg_adata.X = agg_adata.layers['mean']
        # # softmax of over 9 classes
        # from scipy.special import softmax
        X = agg_adata.X.T
        X_norm = X / X.sum(axis=1, keepdims=True)
        X_norm = np.nan_to_num(X_norm, nan=1/9)
        df_xnorm = pd.DataFrame(X_norm, index=agg_adata.var_names, columns=agg_adata.obs_names)
        df_xnorm_safe = df_xnorm.replace(0, 1e-12)
        entropy_values = -np.sum(df_xnorm_safe * np.log(df_xnorm_safe), axis=1)
        # 放回数据框
        df_xnorm['entropy'] = entropy_values
        # pvals = np.array([g_test_row(row) for row in agg_adata.X.T])
        # df_xnorm['pval'] = pvals
        df_xnorm.to_csv(f"/data2st1/junyi/output/atac1112/subset/dmr_region_nt/{region}_{methtype}_celltype_fraction.csv")
        #
        sc.pp.log1p(adata_region)
        sc.tl.rank_genes_groups(adata_region, groupby=group_key, method="wilcoxon",pts=True)
        for ct in adata_region.obs[group_key].unique():
            df_cts = sc.get.rank_genes_groups_df(adata_region, group=ct,pval_cutoff=0.05)
            df_cts.to_csv(f'/data2st1/junyi/output/atac1112/dar/cts/dmr_wilcoxon/{region}_{methtype}_wilcox_{ct}.csv', index=False)
